# 05 — Sensitivity

One-factor-at-a-time sensitivity around the main recipe. The only outputs for
the paper are three ordered curves: topology weight, training/(H_0) batch size,
and the size of the fixed calibration subset reused by every epoch-wise refit.
The default configuration is trained once and reused in all three panels.

In [2]:
# 1. Settings
from pathlib import Path

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO, INSTALL_REQUIREMENTS = True, True
PAIR = "qwen3_0.6b_to_minilm_h384"
TRAIN_DATA_REL = Path("data/train_set/train_100k.csv")
RUN_NAME = f"sensitivity_{PAIR}_v2"
SEEDS = [42]
DEFAULTS = {"lambda": 0.75, "batch": 128, "gauge_samples": 16384}
VALUES = {
    "lambda": [0.0, 0.5, 0.75, 1.0],
    "batch": [16, 64, 128, 256],
    "gauge_samples": [2048, 16384, 32768, 65536],
}
EPOCHS, LR = 5, 7e-5
EXECUTE, STOP_ON_ERROR, REQUIRE_COMPLETE = True, True, True
CUDA_VISIBLE_DEVICES = "0"

In [3]:
# 2. Repo, dependencies, GPU và dữ liệu
import subprocess, sys

cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"

tracked = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"],
    check=True, capture_output=True, text=True,
).stdout.strip()
if AUTO_PULL_REPO and not tracked:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif AUTO_PULL_REPO:
    print("[git] Bỏ qua pull vì repo có tracked changes.")
if INSTALL_REQUIREMENTS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )

git_head = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
if EXECUTE:
    import torch
    assert torch.cuda.is_available(), "Hãy bật GPU runtime trước khi train."
print(f"Repo: {PROJECT_DIR} @ {git_head}")
print(f"Training data: {TRAIN_DATA}")

Repo: /content/embedding-kd @ bc022f3
Training data: /content/embedding-kd/data/train_set/train_100k.csv


In [4]:
# 3. Plan: unique OFAT settings; do not duplicate the default run
import shlex
from _analysis_common import PAIRS, collect_jobs, geoode_command, run_jobs

pair, cache_dir = PAIRS[PAIR], PROJECT_DIR / "runs" / "teacher_cache"
run_root = PROJECT_DIR / "runs" / RUN_NAME
run_root.mkdir(parents=True, exist_ok=True)
specs = [{"arm": "default", **DEFAULTS}]
for sweep, values in VALUES.items():
    for value in values:
        if value == DEFAULTS[sweep]:
            continue
        spec = {"arm": f"{sweep}_{value:g}", **DEFAULTS}
        spec[sweep] = value
        spec["sweep"], spec["value"] = sweep, value
        specs.append(spec)
jobs = []
for spec in specs:
    for seed in SEEDS:
        run_dir = run_root / spec["arm"] / f"seed_{seed}"
        extra = ["--projection_type", "pca", "--gauge_align", "--gauge_rotation", "procrustes",
                 "--gauge_refit_every", 1, "--gauge_align_samples", spec["gauge_samples"],
                 "--lambda_end", 1, "--lambda_ctr", 0, "--lambda_topo", spec["lambda"],
                 "--lambda_h1", 0, "--topo_teacher_source", "original", "--no_eval_retrieval"]
        jobs.append({
            "name": f"{spec['arm']}/seed_{seed}", "arm": spec["arm"], "seed": seed,
            "sweep": spec.get("sweep", "default"), "value": spec.get("value"),
            "run_dir": run_dir,
            "command": geoode_command(
                PROJECT_DIR, pair=pair, train_data=TRAIN_DATA, cache_dir=cache_dir,
                run_dir=run_dir, seed=seed, batch_size=spec["batch"], epochs=EPOCHS,
                learning_rate=LR, extra=extra,
            ),
        })
print(f"Plan: {len(jobs)} unique jobs -> {run_root}")
for job in jobs:
    print(shlex.join(job["command"]))

Plan: 10 unique jobs -> /content/embedding-kd/runs/sensitivity_qwen3_0.6b_to_minilm_h384_v2
/usr/bin/python3 /content/embedding-kd/main.py --method geoode --train_data /content/embedding-kd/data/train_set/train_100k.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --teacher_pooling last_token --batch_size 128 --epochs 5 --save_every 5 --lr 7e-05 --max_length 256 --seed 42 --num_workers 2 --eval_every 0 --pair_threshold_source validation --no-evaluate_test_each_epoch --cache_dir /content/embedding-kd/runs/teacher_cache --save_dir /content/embedding-kd/runs/sensitivity_qwen3_0.6b_to_minilm_h384_v2/default/seed_42 --no_wandb --student_pooling cls --projection_type pca --gauge_align --gauge_rotation procrustes --gauge_refit_every 1 --gauge_align_samples 16384 --lambda_end 1 --lambda_ctr 0 --lambda_topo 0.75 --lambda_h1 0 --topo_teacher_source original --no_eval_retrieval
/usr/bin/python3 /content/embedding-kd/main.py --method 

In [ ]:
# 4. Chạy tuần tự; final-test record là resume boundary
from IPython.display import display

if EXECUTE:
    display(run_jobs(
        PROJECT_DIR, jobs, cuda_visible_devices=CUDA_VISIBLE_DEVICES,
        stop_on_error=STOP_ON_ERROR,
    ))
else:
    print("Dry run: đặt EXECUTE=True để chạy các job còn thiếu.")

[RUN 1/10] default/seed_42 -> /content/embedding-kd/runs/sensitivity_qwen3_0.6b_to_minilm_h384_v2/default/seed_42

Configuration for GEOODE method:
  task_type                 : pair_cls
  max_length                : 256
  batch_size                : 128
  epochs                    : 5
  learning_rate             : 7e-05
  min_lr                    : 2e-06
  warmup_ratio              : 0.06
  w_task                    : 0.5
  alpha_dtw                 : 0.5
  w_cls                     : 1.0
  temperature               : 0.07
  student_model_name        : nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base
  teacher_model_name        : Qwen/Qwen3-Embedding-0.6B
  teacher_dtype             : bfloat16
  pooling_method            : last_token
  student_special_token     : ##
  teacher_special_token     : G
  train_data_path           : /content/embedding-kd/data/train_set/train_100k.csv
  cache_dir                 : /content/embedding-kd/runs/teacher_cache
  cache_batch_size          : 128


In [ ]:
# 5. Appendix Figure A2 — three ordered sensitivity curves
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from _analysis_common import set_paper_style

results = collect_jobs(jobs)
results.to_csv(run_root / "sensitivity_by_run.csv", index=False)
done = results.query("status == 'done'").copy()
if done.empty:
    print("No completed runs yet.")
else:
    expected = len(specs) * len(SEEDS)
    if REQUIRE_COMPLETE and len(done) != expected:
        raise RuntimeError(f"Incomplete sensitivity grid: got {len(done)}, expected {expected}")
    default = done.query("arm == 'default'")[["seed", "avg_all"]]
    rows = []
    for sweep, values in VALUES.items():
        for value in values:
            if value == DEFAULTS[sweep]:
                part = default
            else:
                part = done.query("sweep == @sweep and value == @value")[["seed", "avg_all"]]
            for record in part.itertuples(index=False):
                rows.append({"sweep": sweep, "value": value, "seed": record.seed, "avg_all": record.avg_all})
    curves = pd.DataFrame(rows)
    curves.to_csv(run_root / "sensitivity_curve_values.csv", index=False)
    set_paper_style()
    panels = [
        ("lambda", r"Topology weight $\lambda$", "#2B6CB0"),
        ("batch", r"Training / $H_0$ batch", "#2F855A"),
        ("gauge_samples", "Fixed calibration samples", "#6B46C1"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(5.5, 2.15), sharey=True)
    for index, (ax, (sweep, xlabel, color)) in enumerate(zip(axes, panels)):
        stats = curves.query("sweep == @sweep").groupby("value")["avg_all"].agg(["mean", "std"]).reset_index().sort_values("value")
        x = np.arange(len(stats))
        mean, sd = 100 * stats["mean"].to_numpy(), 100 * stats["std"].fillna(0).to_numpy()
        ax.plot(x, mean, marker="o", ms=3.5, color=color)
        ax.fill_between(x, mean - sd, mean + sd, color=color, alpha=.18, linewidth=0)
        default_x = int(np.flatnonzero(np.isclose(stats.value, DEFAULTS[sweep]))[0])
        ax.axvline(default_x, color="#DD6B20", ls="--", lw=1)
        labels = ([f"{v:g}" for v in stats.value] if sweep == "lambda" else
                  [f"{int(v / 1024)}k" for v in stats.value] if sweep == "gauge_samples" else
                  [f"{int(v)}" for v in stats.value])
        ax.set(xticks=x, xticklabels=labels, xlabel=xlabel, title=f"({chr(97 + index)})")
    axes[0].set_ylabel("Final AVG ×100")
    fig.text(.02, .01, "Mean ± sample SD over 3 seeds; dashed line marks the default.",
             fontsize=6.5, color="#6B7280")
    fig.tight_layout(rect=(0, .06, 1, 1), w_pad=.8)
    fig.savefig(run_root / "figure_A2_sensitivity.pdf", bbox_inches="tight")
    fig.savefig(run_root / "figure_A2_sensitivity.png", dpi=300, bbox_inches="tight")
    plt.show()